# 02. 데이터 정제 2 (Data Cleaning 2)

이 노트북은 일 단위 기록 누락 처리, 빈 시계열 행 보간, 그리고 기간 필터링을 수행하는 데이터 정제 파이프라인의 2단계입니다.

### 주요 공정:
1. **HDD 개체 일 단위 누락 처리**:
   - **Case 1 (고장 후 재기록)**: 마지막이 `failure=1`로 끝났다가 이후 다시 기록이 존재하는 좀비 개체를 색출하여 데이터 전체를 파기합니다.
   - **Case 2 (중도 공백 발생)**: 일 기록 중 1일 공백은 보간(FFill)으로 처리하되, 2일 이상 공백(날짜 차이 >= 3일) 발생 시 공백 전후를 다른 시리얼 넘버(`_1`, `_2` 등)로 분할합니다.
   - **Case 3 (중도 이탈)**: 고장 없이(`failure=0`) 수집 기간 중간에 기록이 중단된 개체의 경우, 라벨 오염 방지를 위해 마지막 30일치 데이터를 삭제합니다.
2. **빈 시계열 채우기 & Forward Fill**: 각 개체별 시계열 내에 비어 있는 일자(Date) 행을 생성하고, 이전 SMART 값들로 Forward Fill을 진행합니다.
3. **기간 필터링**: 분석 신뢰성 확보를 위해 `2014-03-01` 이전 데이터는 전부 제거합니다.
4. **엄밀한 검증 테스트**: 정제 결과물이 모든 제약 조건을 완벽히 준수하는지 테스트 로직을 수행합니다.

## 1. 환경 설정 및 데이터 로드

In [ ]:
import duckdb
import os
import time
import pandas as pd
import numpy as np
from pathlib import Path

data_dir = Path("../data2/01_cleaned")
data_dir.mkdir(parents=True, exist_ok=True)
input_path = (data_dir / "ST4000DM000_cleaned_1.parquet").as_posix()
output_path = (data_dir / "ST4000DM000_cleaned_2.parquet").as_posix()
db_cache_path = (data_dir / "purify_temp.duckdb").as_posix()

print(f"입력 파일: {input_path}")
print(f"출력 파일: {output_path}")

## 2. HDD 개체의 기록 누락 및 이탈 처리 (Case 1, 2, 3)

DuckDB를 활용하여 Case 1, 2, 3 정제 쿼리를 수행합니다.

In [ ]:
print("🚀 HDD 개체 타임라인 정제 시작...")
start_time = time.time()

if os.path.exists(db_cache_path):
    os.remove(db_cache_path)

con = duckdb.connect(db_cache_path)
con.execute("PRAGMA memory_limit='4GB'")
con.execute("PRAGMA temp_directory='duckdb_temp'")

try:
    # input_path (cleaned_1.parquet)의 date는 이미 01 노트북에서 DATE 타입으로 저장되었으므로 캐스팅 불필요.
    # 단, Case 1, 2 쿼리 내 연산 정합성을 위해 그대로 컬럼명을 사용합니다.
    
    # [Case 1] 고장 발생(failure=1) 이후에 다시 정상 데이터가 기록되는 좀비 디스크를 찾아 제거
    # (참고: input_path에는 이미 D-30 ~ D-1 구간이 failure=1로 라벨링되어 있으므로 고장 당일(D-DAY) 역시 failure=1입니다)
    print("-> [Case 1] 고장 후 재기록 개체 색출 및 제거 중...")
    con.execute(f"""
        CREATE TABLE t1_clean AS 
        WITH GhostSerials AS (
            SELECT serial_number
            FROM read_parquet('{input_path}')
            GROUP BY serial_number
            HAVING MAX(CASE WHEN failure = 1 THEN date ELSE NULL END) < MAX(date)
        )
        SELECT *
        FROM read_parquet('{input_path}')
        WHERE serial_number NOT IN (SELECT serial_number FROM GhostSerials);
    """)
    
    # [Case 2] 2일 이상 공백(날짜 차이 >= 3일) 발생 시 다른 시리얼 넘버(_1, _2)로 쪼개기
    print("-> [Case 2] 2일 이상 시계열 공백 개체 분할 중...")
    con.execute(f"""
        CREATE TABLE t2_chunked AS 
        WITH LaggedData AS (
            SELECT *,
                   LAG(date) OVER (PARTITION BY serial_number ORDER BY date) as prev_date
            FROM t1_clean
        ),
        GapFlagged AS (
            SELECT *,
                   CASE WHEN prev_date IS NOT NULL 
                         AND date_diff('day', prev_date, date) >= 3 
                        THEN 1 ELSE 0 END as gap_flag
            FROM LaggedData
        ),
        ChunkedData AS (
            SELECT *,
                   SUM(gap_flag) OVER (PARTITION BY serial_number ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as chunk_id
            FROM GapFlagged
        ) 
        SELECT * EXCLUDE(prev_date, gap_flag),
               CASE WHEN chunk_id = 0 THEN serial_number 
                    ELSE serial_number || '_' || CAST(chunk_id AS VARCHAR) 
               END as new_serial
        FROM ChunkedData;
    """)

    # [Case 3] 고장 없이(seq_failure_status = 0) 중간에 중단된 개체의 마지막 30일 데이터 삭제
    print("-> [Case 3] 중도 이탈 개체의 마지막 30일 데이터 파기 중...")
    con.execute("""
        CREATE TABLE t3_trimmed AS 
        WITH GlobalMax AS (
            SELECT MAX(CAST(date AS DATE)) as g_max_date FROM t2_chunked
        ),
        SeqEndings AS (
            SELECT new_serial, 
                   MAX(CAST(date AS DATE)) as seq_max_date,
                   MAX(failure) as seq_failure_status
            FROM t2_chunked
            GROUP BY new_serial
        ),
        ToDrop30Days AS (
            SELECT s.new_serial, s.seq_max_date
            FROM SeqEndings s
            CROSS JOIN GlobalMax g
            WHERE s.seq_failure_status = 0 
              AND s.seq_max_date < g.g_max_date
        )
        SELECT s.* EXCLUDE(chunk_id)
        FROM t2_chunked s
        LEFT JOIN ToDrop30Days d ON s.new_serial = d.new_serial
        WHERE d.new_serial IS NULL 
           OR CAST(s.date AS DATE) <= d.seq_max_date - INTERVAL 30 DAY;
    """)
    
    print(f"✅ HDD 개체 타임라인 정제 완료! (소요 시간: {time.time() - start_time:.2f}초)")
except Exception as e:
    print(f"❌ 오류 발생: {e}")
finally:
    con.close()

## 3. 빈 시계열 행 생성 및 일괄 Forward Fill

각 개체의 연속적인 모니터링을 위해 비어 있는 일자 행을 생성하고, 이전 시점의 값으로 채워 넣습니다.

In [ ]:
print("🚀 1일 단위 빈 시계열 생성 및 FFill 적용 중...")
start_time = time.time()

con = duckdb.connect(db_cache_path)
con.execute("PRAGMA memory_limit='4GB'")
con.execute("PRAGMA temp_directory='duckdb_temp'")

try:
    cols_df = con.execute("DESCRIBE SELECT * FROM t3_trimmed").fetchdf()
    feature_cols = [c for c in cols_df['column_name'].tolist() if c not in ['serial_number', 'new_serial', 'date', 'failure']]

    ffill_sqls = []
    for col in feature_cols:
         ffill_sqls.append(f"LAST_VALUE(t.\"{col}\" IGNORE NULLS) OVER (PARTITION BY c.new_serial ORDER BY c.date) as \"{col}\"")
    ffill_str = ",\n                ".join(ffill_sqls)

    # 달력을 생성하여 빈 날짜를 채우고, FFill 적용
    con.execute(f"""
        CREATE TABLE t4_ffilled AS 
        WITH BoundaryDates AS (
            SELECT new_serial, MIN(CAST(date AS DATE)) as min_date, MAX(CAST(date AS DATE)) as max_date
            FROM t3_trimmed
            GROUP BY new_serial
        ),
        Calendar AS (
            SELECT new_serial, 
                   CAST(UNNEST(generate_series(min_date, max_date, INTERVAL 1 DAY)) AS DATE) as date
            FROM BoundaryDates
        )
        SELECT 
            c.new_serial as serial_number,
            c.date,
            COALESCE(t.failure, 0) as failure,
            {ffill_str}
        FROM Calendar c
        LEFT JOIN t3_trimmed t 
          ON c.new_serial = t.new_serial 
         AND c.date = CAST(t.date AS DATE)
    """)
    print(f"✅ FFill 연산 완료! (소요 시간: {time.time() - start_time:.2f}초)")
finally:
    con.close()

## 4. 기간 필터링 및 최종 파일 저장

데이터 신뢰성이 충분히 확보된 시점인 `2014-03-01` 이후의 레코드만 보존하여 최종 Parquet 파일로 출력합니다.

In [ ]:
print("🚀 2014-03-01 이전 데이터 필터링 및 최종 저장 중...")
start_time = time.time()

con = duckdb.connect(db_cache_path)
try:
    con.execute(f"""
        COPY (
            SELECT * 
            FROM t4_ffilled
            WHERE date >= '2014-03-01'
            ORDER BY serial_number, date
        ) TO '{output_path}' (FORMAT PARQUET);
    """)
    print(f"✅ 저장 완료! (소요 시간: {time.time() - start_time:.2f}초)")
    print(f"📦 저장 경로: {output_path}")
finally:
    con.close()
    if os.path.exists(db_cache_path):
        os.remove(db_cache_path)

## 5. 엄밀한 전처리 정합성 검증 테스트 (Verification Tests)

정제된 데이터셋이 명세된 전처리 규칙들을 단 하나도 위반하지 않았는지 엄격하게 검증합니다.

In [10]:
print("🔍 [통합성 검증] 정제 결과물 정밀 테스트 시작...")
con = duckdb.connect()

try:
    safe_date_cast = "COALESCE(TRY_CAST(date AS DATE), strptime(date, '%m/%d/%y')::DATE)"

    # 1. 일자 범위 검증
    print("Test 1: 2014-03-01 이전 데이터 검출 테스트")
    pre_2014 = con.execute(f"SELECT COUNT(*) FROM read_parquet('{output_path}') WHERE date < '2014-03-01'").fetchone()[0]
    assert pre_2014 == 0, f"오류: 2014-03-01 이전 데이터가 {pre_2014}건 존재합니다"
    print("  -> [PASS] 2014-03-01 이전 데이터 없음.")

    # 2. Case 1 검증: 고장 발생(D-DAY) 일자 이후로도 기록이 이어서 나타나는 좀비 개체가 존재하지 않는지 검증
    # (참고: 라벨링된 데이터셋의 failure=1은 D-30부터 시작하므로, 실제 고장일 D-DAY는 raw 데이터셋의 failure=1 날짜와 대칭 매칭해야 정확합니다)
    print("Test 2: [Case 1] 고장 발생 후 뒤이어 개체 존재 여부 검증")
    zombie_count = con.execute(f"""
        WITH FailDates AS (
            SELECT serial_number, MIN({safe_date_cast}) as fail_date
            FROM read_parquet('../data2/01_cleaned/ST4000DM000_raw.parquet')
            WHERE failure = 1
            GROUP BY serial_number
        )
        SELECT COUNT(DISTINCT REGEXP_REPLACE(t.serial_number, '_[0-9]+$', ''))
        FROM read_parquet('{output_path}') t
        JOIN FailDates f ON REGEXP_REPLACE(t.serial_number, '_[0-9]+$', '') = f.serial_number
        WHERE t.date > f.fail_date
    """).fetchone()[0]
    assert zombie_count == 0, f"오류: 고장일 이후에도 로그가 이어지는 개체가 {zombie_count}개 존재합니다"
    print("  -> [PASS] 고장(failure=1) 후 뒤이어 개체 없음.")

    # 3. Case 2 검증: 동일 serial_number 안에 2일 이상의 비구간(일자 차이 >= 3일) 사전에 존재하는지 확인
    print("Test 3: [Case 2] 2일 이상 공백구간 미제거 개체 검출 테스트")
    gap_errors = con.execute(f"""
        WITH Lagged AS (
            SELECT serial_number, date,
                   LAG(date) OVER (PARTITION BY serial_number ORDER BY date) as prev_date
            FROM read_parquet('{output_path}')
        )
        SELECT COUNT(*)
        FROM Lagged
        WHERE prev_date IS NOT NULL 
          AND date_diff('day', CAST(prev_date AS DATE), CAST(date AS DATE)) >= 3
    """).fetchone()[0]
    assert gap_errors == 0, f"오류: 2일 이상의 일자 갭이 발견된 개체가 {gap_errors}건 존재합니다"
    print("  -> [PASS] 2일 이상의 끊락 공백 없이 정상 분할/보간됨")

    # 4. Case 3 검증: 중도 이탈 디스크 말미 30일치 데이터 절단 검증
    print("Test 4: [Case 3] 중도 이탈 개체의 말미 30일 제거 통합성 검증")
    mismatches = con.execute(f"""
        WITH GlobalMax AS (
            SELECT MAX(CAST(date AS DATE)) as g_max_date FROM read_parquet('{input_path}')
        ),
        OrigEndings AS (
            SELECT serial_number, 
                   MAX(CAST(date AS DATE)) as orig_max_date,
                   MAX(failure) as orig_fail
            FROM read_parquet('{input_path}')
            GROUP BY serial_number
        ),
        NewEndings AS (
            SELECT REGEXP_REPLACE(serial_number, '_[0-9]+$', '') as serial_number, 
                   MAX(CAST(date AS DATE)) as new_max_date
            FROM read_parquet('{output_path}')
            GROUP BY REGEXP_REPLACE(serial_number, '_[0-9]+$', '')
        )
        SELECT COUNT(*)
        FROM OrigEndings o
        JOIN NewEndings n ON o.serial_number = n.serial_number
        CROSS JOIN GlobalMax g
        WHERE o.orig_fail = 0 
          AND o.orig_max_date < g.g_max_date
          AND date_diff('day', n.new_max_date, o.orig_max_date) < 30
          AND o.orig_max_date >= '2014-03-01'
    """).fetchone()[0]
    assert mismatches == 0, f"오류: 중도 이탈 개체 중 30일치가 절제되지 않은 개체가 {mismatches}개 존재합니다"
    print("  -> [PASS] 중도 이탈 개체의 말미 30일 데이터 정상 절단 완료.")

    # 5. 비어있는 일자 채우기 검증
    print("Test 5: 개체 단위 일자 Gap 존재 여부 검증")
    gap_days = con.execute(f"""
        SELECT COUNT(*)
        FROM (
            SELECT serial_number, 
                   MIN(CAST(date AS DATE)) as min_d, 
                   MAX(CAST(date AS DATE)) as max_d,
                   COUNT(*) as row_count
            FROM read_parquet('{output_path}')
            GROUP BY serial_number
        )
        WHERE date_diff('day', min_d, max_d) + 1 <> row_count
    """).fetchone()[0]
    assert gap_days == 0, f"오류: 시리즈 안에 비어있는 일자를 갖고 있는 개체가 {gap_days}개 존재합니다"
    print("  -> [PASS] 개체별 완전하게 1일 단위 연속성을 가진 일자열 정보.")

    # ── 강화 6: 총 row 수 > 0 검증 ──
    print("Test 6: 출력 파일 비어있지 않음 검증")
    total_rows = con.execute(f"SELECT COUNT(*) FROM read_parquet('{output_path}')").fetchone()[0]
    assert total_rows > 0, "오류: 정제 결과 파일이 비어 있습니다"
    print(f"  -> [PASS] 총 {total_rows:,} rows 존재 확인.")

    # ── 강화 7: NaN 존재 여부 검증 ──
    print("Test 7: NaN/Inf 잔존 검증")
    out_cols = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{output_path}')").fetchdf()
    numeric_cols = [r['column_name'] for _, r in out_cols.iterrows() if r['column_name'] not in ('serial_number', 'date') and 'VARCHAR' not in r['column_type']]
    nan_checks = [f"SUM(CASE WHEN isnan({nc}) OR isinf({nc}) THEN 1 ELSE 0 END)" for nc in numeric_cols]
    if nan_checks:
        nan_results = con.execute(f"SELECT {', '.join(nan_checks)} FROM read_parquet('{output_path}')").fetchone()
        bad_cols = [numeric_cols[j] for j, v in enumerate(nan_results) if v and v > 0]
        assert len(bad_cols) == 0, f"오류: NaN/Inf 잔존 컬럼: {bad_cols}"
    print("  -> [PASS] 모든 수치 컬럼에 NaN/Inf 없음 확인.")

    # ── 강화 8: failure 컬럼 값 범위 검증 ──
    print("Test 8: failure 컬럼 값 범위 검증")
    fail_vals = con.execute(f"SELECT DISTINCT failure FROM read_parquet('{output_path}') ORDER BY failure").fetchall()
    fail_set = set(r[0] for r in fail_vals)
    assert fail_set.issubset({0, 1}), f"오류: failure 컬럼에 0/1 이외 값 존재: {fail_set}"
    print("  -> [PASS] failure 컬럼 값이 {0, 1}만 존재 확인.")

    # ── 강화 9: 정제 전후 고장 개체 수 보존 검증 ──
    # (4단계 기간 필터링 'date >= 2014-03-01'에 의해 2014년 3월 이전에 고장난 개체는 제외되는 것이 타당하므로, 대조 대상을 해당 기간 이후 고장 개체로 보정해 검증합니다)
    print("Test 9: 정제 전후 고장 개체 수 보존 검증")
    con.execute(f"""
        CREATE TEMP TABLE temp_orig_fail AS 
        SELECT DISTINCT serial_number 
        FROM read_parquet('{input_path}') 
        WHERE date >= '2014-03-01' 
          AND failure = 1
          AND serial_number NOT IN (
              SELECT serial_number 
              FROM read_parquet('{input_path}') 
              GROUP BY serial_number 
              HAVING MAX(CASE WHEN failure = 1 THEN date ELSE NULL END) < MAX(date)
          )
    """)
    orig_fail_cnt = con.execute("SELECT COUNT(*) FROM temp_orig_fail").fetchone()[0]
    con.execute("DROP TABLE temp_orig_fail")
    
    new_fail_cnt = con.execute(f"SELECT COUNT(DISTINCT REGEXP_REPLACE(serial_number, '_[0-9]+$', '')) FROM read_parquet('{output_path}') WHERE failure = 1").fetchone()[0]
    assert orig_fail_cnt == new_fail_cnt, f"오류: 고장 개체 수 불일치! 원본={orig_fail_cnt}, 정제후={new_fail_cnt}"
    print(f"  -> [PASS] 고장 개체 수 {new_fail_cnt}개 완전 보존 확인.")

    print("\n✅ [통합성 검증 완료] 모든 정밀 테스트 조건을 만족합니다 (9/9 PASS)")
finally:
    con.close()


🔍 [통합성 검증] 정제 결과물 정밀 테스트 시작...
Test 1: 2014-03-01 이전 데이터 검출 테스트
  -> [PASS] 2014-03-01 이전 데이터 없음.
Test 2: [Case 1] 고장 발생 후 뒤이어 개체 존재 여부 검증
  -> [PASS] 고장(failure=1) 후 뒤이어 개체 없음.
Test 3: [Case 2] 2일 이상 공백구간 미제거 개체 검출 테스트
  -> [PASS] 2일 이상의 끊락 공백 없이 정상 분할/보간됨
Test 4: [Case 3] 중도 이탈 개체의 말미 30일 제거 통합성 검증
  -> [PASS] 중도 이탈 개체의 말미 30일 데이터 정상 절단 완료.
Test 5: 개체 단위 일자 Gap 존재 여부 검증
  -> [PASS] 개체별 완전하게 1일 단위 연속성을 가진 일자열 정보.
Test 6: 출력 파일 비어있지 않음 검증
  -> [PASS] 총 77,362,978 rows 존재 확인.
Test 7: NaN/Inf 잔존 검증
  -> [PASS] 모든 수치 컬럼에 NaN/Inf 없음 확인.
Test 8: failure 컬럼 값 범위 검증
  -> [PASS] failure 컬럼 값이 {0, 1}만 존재 확인.
Test 9: 정제 전후 고장 개체 수 보존 검증
  -> [PASS] 고장 개체 수 5701개 완전 보존 확인.

✅ [통합성 검증 완료] 모든 정밀 테스트 조건을 만족합니다 (9/9 PASS)
